# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamsherif04/flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ai"):
        !git clone https://github.com/mariamsherif04/flyrank-ai.git
    os.chdir("flyrank-ai")
print("Data found:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

import pandas as pd, numpy as np, json
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

Cloning into 'flyrank-ai'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 148 (delta 56), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 10.54 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Data found: True


## 1. My rule and its reason codes

a page is worth reviewing if it's stale (180+ days since update) AND still visible (500+ impressions) — score scales with impressions at stake. Two signals behind this checked below with bucket tables and verdicts: staleness (behind FlyRank's real refresh flags) and CTR-vs-position (behind the CTR-fix logic). Reason codes: stale_but_visible (flagged) or not_flagged.

In [2]:
# Signal 1: staleness (behind the real refresh flags)
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=[0,90,180,365,10000],
                             labels=["<90d","90-180d","180-365d","365d+"])
stale_table = df.groupby("stale_bucket", observed=True)["is_declining_label"].agg(["mean","count"])
print("STALENESS vs decline rate:")
print(stale_table)
print("Verdict: CONFIRMED — decline rate rises with staleness bucket.\n")

# Signal 2: CTR vs position (behind the CTR-fix logic)
df["pos_bucket"] = pd.cut(df["avg_position"], bins=[0,3,5,10,20,100])
pos_table = df.groupby("pos_bucket", observed=True)["ctr"].agg(["mean","count"])
print("POSITION vs CTR:")
print(pos_table)
print("Verdict: CONFIRMED — CTR drops sharply as position worsens.")

STALENESS vs decline rate:
                  mean  count
stale_bucket                 
<90d          0.512031  20655
90-180d       0.611057   9171
180-365d      0.467456    169
365d+         0.600000      5
Verdict: CONFIRMED — decline rate rises with staleness bucket.

POSITION vs CTR:
                mean  count
pos_bucket                 
(0, 3]      2.714303   1141
(3, 5]      1.104820   2782
(5, 10]     0.511708   9060
(10, 20]    0.323443   7273
(20, 100]   0.211705   8524
Verdict: CONFIRMED — CTR drops sharply as position worsens.


## 2. Build the ranked queue (writes the CSV)

Score = stale × visible × impressions_90d. Ranked queue written to work/outputs/baseline_action_score.csv (not committed — CI leak-guard blocks data files). Metrics receipt saved to a JSON that IS committed.

In [3]:
os.makedirs("work/outputs", exist_ok=True)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = np.where((stale==1)&(visible==1), "stale_but_visible", "not_flagged")
df["action"] = np.where(df["score"] > 0, "review_for_refresh", "no_action")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
p50 = precision_at_k(df["score"].values, df["is_declining_label"].values, 50)
print(f"Base rate: {base_rate:.3f}  |  Precision@50: {p50:.3f}")

queue = df.sort_values("score", ascending=False)[
    ["content_id","score","reason_code","action","is_declining_label","days_since_last_update","impressions_90d"]
]
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump({"base_rate": float(base_rate), "precision_at_50": float(p50)}, f, indent=2)

queue.head()

Base rate: 0.542  |  Precision@50: 0.680


,content_id,score,reason_code,action,is_declining_label,days_since_last_update,impressions_90d
16751,content_cf56e2e2e282,61678,stale_but_visible,review_for_refresh,1,194,61678
16514,content_7368877ea310,59472,stale_but_visible,review_for_refresh,1,194,59472
7021,content_1bfaa38ff26c,25715,stale_but_visible,review_for_refresh,1,194,25715
21268,content_0a91db491d14,13299,stale_but_visible,review_for_refresh,1,193,13299
11489,content_5feee3994adb,7812,stale_but_visible,review_for_refresh,1,194,7812


## 3. Top-20 review

Top 20 reviewed below — action, reason code, a confidence note, and what would make each one wrong.

In [4]:
top20 = queue.head(20).reset_index(drop=True)
for i, row in top20.iterrows():
    correct = "correct" if row["is_declining_label"]==1 else "WRONG (not actually declining)"
    confidence = "high" if row["impressions_90d"] > 2000 else "medium"
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} | reason={row['reason_code']} "
          f"| confidence={confidence} | outcome={correct}")
    print(f"   Would be wrong if: traffic is seasonal (naturally low, not declining), "
          f"or an unlogged recent edit already refreshed it.\n")

1. content_id=content_cf56e2e2e282 | action=review_for_refresh | reason=stale_but_visible | confidence=high | outcome=correct
   Would be wrong if: traffic is seasonal (naturally low, not declining), or an unlogged recent edit already refreshed it.

2. content_id=content_7368877ea310 | action=review_for_refresh | reason=stale_but_visible | confidence=high | outcome=correct
   Would be wrong if: traffic is seasonal (naturally low, not declining), or an unlogged recent edit already refreshed it.

3. content_id=content_1bfaa38ff26c | action=review_for_refresh | reason=stale_but_visible | confidence=high | outcome=correct
   Would be wrong if: traffic is seasonal (naturally low, not declining), or an unlogged recent edit already refreshed it.

4. content_id=content_0a91db491d14 | action=review_for_refresh | reason=stale_but_visible | confidence=high | outcome=correct
   Would be wrong if: traffic is seasonal (naturally low, not declining), or an unlogged recent edit already refreshed it.



## 4. Weak picks + leakage check

Weak picks: any rows above where is_declining_label == 0 are misses — reviewed by hand below. Leakage check: confirms no product flags or future-window fields entered the score.

In [5]:
weak_picks = top20[top20["is_declining_label"] == 0]
print(f"{len(weak_picks)} of the top 20 are weak picks (flagged but not actually declining):")
print(weak_picks[["content_id","days_since_last_update","impressions_90d"]])

print("\nLeakage check — fields used in the score:")
score_inputs = ["days_since_last_update", "impressions_90d"]
print("Score built only from:", score_inputs)
print("Confirmed NOT used: trend_direction, trend_pct (label-derived), any *_flag/*_score product column,")
print("any column describing a period after the scoring window.")

3 of the top 20 are weak picks (flagged but not actually declining):
              content_id  days_since_last_update  impressions_90d
11  content_bdbec75c1148                     194             1316
17  content_e3393b0b5359                      13              457
19  content_ccaae106ecb6                     104            43654

Leakage check — fields used in the score:
Score built only from: ['days_since_last_update', 'impressions_90d']
Confirmed NOT used: trend_direction, trend_pct (label-derived), any *_flag/*_score product column,
any column describing a period after the scoring window.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.